# Этап 1. Сбор датасета

Вход: seed-список (встроен в ноутбук)

Выход: `data/tracks.csv`, `audio/*.wav`

In [ ]:
%pip install -q "yt-dlp[default]" imageio-ffmpeg rapidfuzz librosa soundfile umap-learn transformers tqdm
%pip install -q deno

In [ ]:
import os, sys, json, time, re, subprocess, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 80)

IN_COLAB = 'google.colab' in sys.modules
USE_DRIVE = True

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/music_dataset')
else:
    BASE_DIR = Path('music_dataset').resolve()

AUDIO_DIR, DATA_DIR, FIG_DIR = BASE_DIR / 'audio', BASE_DIR / 'data', BASE_DIR / 'figures'
for d in (AUDIO_DIR, DATA_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)

GENRE_ORDER = ['rock', 'pop', 'hip_hop', 'electronic', 'jazz', 'classical',
               'metal', 'rnb_soul', 'country', 'reggae', 'ambient', 'latin']
GENRE_RU = {'rock': 'рок', 'pop': 'поп', 'hip_hop': 'хип-хоп', 'electronic': 'электроника',
            'jazz': 'джаз', 'classical': 'классика', 'metal': 'метал', 'rnb_soul': 'R&B / соул',
            'country': 'кантри', 'reggae': 'регги', 'ambient': 'эмбиент', 'latin': 'латино'}

## 1. Сбор датасета

**Источники.** Аудио и метаданные я беру с YouTube через `yt-dlp`: у почти всех известных треков там есть официальная загрузка (каналы `... - Topic`, VEVO или канал самого артиста). С YouTube же забираю длительность, количество просмотров и лайков, дату загрузки и теги. Год выпуска дополнительно сверяю с MusicBrainz, это открытая музыкальная база с API.

**Как отбирала треки.** Список составила вручную: 12 жанров, в каждом по 8 известных треков. Брала узнаваемые записи, чтобы на YouTube точно нашлась оригинальная версия, а не кавер. Внутри жанра старалась взять треки из разных десятилетий, иначе эпоха полностью совпадала бы с жанром.

Видео для каждого трека подбирается автоматически: ищу на YouTube и выбираю результат, в котором совпадают название и исполнитель, канал официальный и в заголовке нет слов cover, remix, live, karaoke и т. п. Потом спорные случаи проверяю глазами.

Из каждого трека вырезаю 30 секунд, начиная примерно с трети длины (там обычно уже куплет или припев, а не вступление). Сохраняю в WAV 48 кГц моно, потому что с такой частотой работает CLAP.

**Жанры:** рок, поп, хип-хоп, электроника, джаз, классика, метал, R&B/соул, кантри, регги, эмбиент, латино.

**Ограничения:**
- 8 треков на жанр, всего 96, чтобы классы были сбалансированы;
- один трек на исполнителя, иначе при поиске похожих треков модель находила бы просто того же артиста;
- у классики в качестве года стоит год написания произведения, а не год записи.

In [ ]:
import io
MAX_PER_GENRE, MAX_PER_ARTIST = 8, 1

SEED_CSV = """track_id,artist,title,genre,year,query
rock_01,Queen,Bohemian Rhapsody,rock,1975,
rock_02,Led Zeppelin,Stairway to Heaven,rock,1971,
rock_03,Nirvana,Smells Like Teen Spirit,rock,1991,
rock_04,The Rolling Stones,Paint It Black,rock,1966,
rock_05,Arctic Monkeys,Do I Wanna Know?,rock,2013,
rock_06,Radiohead,Creep,rock,1992,
rock_07,The Killers,Mr. Brightside,rock,2003,
rock_08,Foo Fighters,Everlong,rock,1997,
pop_01,Michael Jackson,Billie Jean,pop,1982,
pop_02,Madonna,Like a Prayer,pop,1989,
pop_03,Britney Spears,Toxic,pop,2003,
pop_04,Dua Lipa,Levitating,pop,2020,
pop_05,The Weeknd,Blinding Lights,pop,2019,
pop_06,Taylor Swift,Shake It Off,pop,2014,
pop_07,ABBA,Dancing Queen,pop,1976,
pop_08,Billie Eilish,bad guy,pop,2019,
hip_hop_01,Dr. Dre,Still D.R.E. (feat. Snoop Dogg),hip_hop,1999,
hip_hop_02,Eminem,Lose Yourself,hip_hop,2002,
hip_hop_03,Kendrick Lamar,HUMBLE.,hip_hop,2017,
hip_hop_04,The Notorious B.I.G.,Juicy,hip_hop,1994,
hip_hop_05,2Pac,California Love (feat. Dr. Dre),hip_hop,1995,
hip_hop_06,OutKast,Ms. Jackson,hip_hop,2000,
hip_hop_07,Run-DMC,It's Tricky,hip_hop,1986,
hip_hop_08,Travis Scott,SICKO MODE,hip_hop,2018,
electronic_01,Daft Punk,One More Time,electronic,2000,
electronic_02,Avicii,Levels,electronic,2011,
electronic_03,The Prodigy,Firestarter,electronic,1996,
electronic_04,deadmau5,Strobe,electronic,2009,
electronic_05,The Chemical Brothers,Hey Boy Hey Girl,electronic,1999,
electronic_06,Kraftwerk,The Model,electronic,1978,
electronic_07,Swedish House Mafia,Don't You Worry Child,electronic,2012,
electronic_08,Fred again..,Delilah (pull me out of this),electronic,2022,
jazz_01,Miles Davis,So What,jazz,1959,
jazz_02,The Dave Brubeck Quartet,Take Five,jazz,1959,
jazz_03,John Coltrane,Giant Steps,jazz,1960,
jazz_04,Louis Armstrong,What a Wonderful World,jazz,1967,
jazz_05,Ella Fitzgerald,Mack the Knife (Live in Berlin),jazz,1960,
jazz_06,Herbie Hancock,Cantaloupe Island,jazz,1964,
jazz_07,Chet Baker,My Funny Valentine,jazz,1954,
jazz_08,Kamasi Washington,Truth,jazz,2017,
classical_01,Ludwig van Beethoven,Symphony No. 5 in C minor: I. Allegro con brio,classical,1808,Beethoven Symphony No. 5 1st movement Allegro con brio
classical_02,Wolfgang Amadeus Mozart,Eine kleine Nachtmusik: I. Allegro,classical,1787,Mozart Eine kleine Nachtmusik I. Allegro
classical_03,Antonio Vivaldi,The Four Seasons - Spring: I. Allegro,classical,1725,Vivaldi Four Seasons Spring I. Allegro
classical_04,Johann Sebastian Bach,Cello Suite No. 1 in G major: Prelude,classical,1720,Bach Cello Suite No. 1 Prelude
classical_05,Claude Debussy,Clair de lune,classical,1905,Debussy Clair de lune piano
classical_06,Pyotr Ilyich Tchaikovsky,The Nutcracker: Dance of the Sugar Plum Fairy,classical,1892,Tchaikovsky Nutcracker Dance of the Sugar Plum Fairy
classical_07,Frédéric Chopin,"Nocturne in E-flat major, Op. 9 No. 2",classical,1832,Chopin Nocturne Op. 9 No. 2
classical_08,Erik Satie,Gymnopédie No. 1,classical,1888,Satie Gymnopedie No. 1
metal_01,Metallica,Master of Puppets,metal,1986,
metal_02,Black Sabbath,Paranoid,metal,1970,
metal_03,Iron Maiden,The Trooper,metal,1983,
metal_04,Slayer,Raining Blood,metal,1986,
metal_05,System of a Down,Chop Suey!,metal,2001,
metal_06,Rammstein,Du hast,metal,1997,
metal_07,Slipknot,Duality,metal,2004,
metal_08,Gojira,Stranded,metal,2017,
rnb_soul_01,Marvin Gaye,What's Going On,rnb_soul,1971,
rnb_soul_02,Aretha Franklin,Respect,rnb_soul,1967,
rnb_soul_03,Stevie Wonder,Superstition,rnb_soul,1972,
rnb_soul_04,Whitney Houston,I Wanna Dance with Somebody (Who Loves Me),rnb_soul,1987,
rnb_soul_05,Destiny's Child,Say My Name,rnb_soul,1999,
rnb_soul_06,Frank Ocean,Thinkin Bout You,rnb_soul,2012,
rnb_soul_07,SZA,Kill Bill,rnb_soul,2022,
rnb_soul_08,Alicia Keys,Fallin',rnb_soul,2001,
country_01,Johnny Cash,Ring of Fire,country,1963,
country_02,Dolly Parton,Jolene,country,1973,
country_03,Kenny Rogers,The Gambler,country,1978,
country_04,John Denver,"Take Me Home, Country Roads",country,1971,
country_05,Shania Twain,Man! I Feel Like a Woman!,country,1997,
country_06,Luke Combs,Fast Car,country,2023,
country_07,Chris Stapleton,Tennessee Whiskey,country,2015,
country_08,Willie Nelson,On the Road Again,country,1980,
reggae_01,Bob Marley & The Wailers,Three Little Birds,reggae,1977,
reggae_02,Peter Tosh,Legalize It,reggae,1976,
reggae_03,Jimmy Cliff,The Harder They Come,reggae,1972,
reggae_04,Toots & The Maytals,Pressure Drop,reggae,1969,
reggae_05,UB40,Red Red Wine,reggae,1983,
reggae_06,Damian Marley,Welcome to Jamrock,reggae,2005,
reggae_07,Burning Spear,Marcus Garvey,reggae,1975,
reggae_08,Chronixx,Here Comes Trouble,reggae,2014,
ambient_01,Brian Eno,An Ending (Ascent),ambient,1983,
ambient_02,Aphex Twin,Rhubarb,ambient,1994,Aphex Twin Rhubarb Selected Ambient Works Volume II
ambient_03,Boards of Canada,Roygbiv,ambient,1998,
ambient_04,Max Richter,On the Nature of Daylight,ambient,2004,
ambient_05,Marconi Union,Weightless,ambient,2011,
ambient_06,Hiroshi Yoshimura,Water Copy,ambient,1982,
ambient_07,Jon Hopkins,Abandon Window,ambient,2013,
ambient_08,Tycho,Awake,ambient,2014,
latin_01,Luis Fonsi,Despacito (feat. Daddy Yankee),latin,2017,
latin_02,Buena Vista Social Club,Chan Chan,latin,1997,
latin_03,Shakira,Hips Don't Lie (feat. Wyclef Jean),latin,2006,
latin_04,Santana,Oye Como Va,latin,1970,
latin_05,Bad Bunny,Tití Me Preguntó,latin,2022,
latin_06,Celia Cruz,La Vida Es Un Carnaval,latin,1998,
latin_07,Stan Getz & João Gilberto,The Girl from Ipanema,latin,1964,
latin_08,Gipsy Kings,Bamboléo,latin,1987,
"""
seed = pd.read_csv(io.StringIO(SEED_CSV), dtype={'query': str})
seed['query'] = seed['query'].fillna('')

assert seed.groupby('genre').size().max() <= MAX_PER_GENRE
assert seed.groupby('artist').size().max() <= MAX_PER_ARTIST
assert seed['track_id'].is_unique
print(len(seed), 'треков,', seed.genre.nunique(), 'жанров,', seed.artist.nunique(), 'исполнителей')
seed.groupby('genre').size().rename(index=GENRE_RU)

### Загрузка с YouTube
Если YouTube из Colab начнёт просить подтверждение («Sign in to confirm you're not a bot»), можно положить cookies из браузера в `cookies.txt` и указать путь в `COOKIES_FILE`. Другой вариант: запустить эту часть локально. Уже скачанные треки повторно не грузятся.

In [ ]:
import yt_dlp, imageio_ffmpeg
from rapidfuzz import fuzz
from tqdm.auto import tqdm

FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()
COOKIES_FILE = None
CLIP_SEC = 30
CLIP_POS = 0.35
SLEEP_SEC = 1.5

BAD_WORDS = ['cover', 'karaoke', 'remix', 'live', 'reaction', 'lyrics', '8d', 'slowed', 'sped up',
             'nightcore', 'instrumental', 'tutorial', 'lesson', 'piano version', 'loop', '1 hour', '10 hours']

def ydl_opts(**extra):
    opts = dict(quiet=True, no_warnings=True, noplaylist=True, ffmpeg_location=FFMPEG)
    if COOKIES_FILE:
        opts['cookiefile'] = COOKIES_FILE
    opts.update(extra)
    return opts

def search_youtube(query, n=6):
    with yt_dlp.YoutubeDL(ydl_opts(extract_flat=True)) as ydl:
        res = ydl.extract_info(f'ytsearch{n}:{query}', download=False)
    return [e for e in (res.get('entries') or []) if e and e.get('id')]

def score_candidate(e, ref_text, artist, title):
    vt = (e.get('title') or '').lower()
    ch = (e.get('channel') or e.get('uploader') or '').lower()
    s = fuzz.token_set_ratio(ref_text.lower(), f'{ch} {vt}')
    if ch.endswith(' - topic'):
        s += 15
    if 'vevo' in ch or fuzz.partial_ratio(artist.lower(), ch) >= 85:
        s += 10
    if 'official' in vt:
        s += 5
    for w in BAD_WORDS:
        if w in vt and w not in title.lower():
            s -= 25
    d = e.get('duration') or 0
    if d and not (60 <= d <= 1200):
        s -= 40
    return s

def pick_video(row):
    if row.track_id in MANUAL_OVERRIDES:
        return {'id': MANUAL_OVERRIDES[row.track_id]}, 100.0
    query = row.query.strip() or f'{row.artist} - {row.title} official audio'
    ref = row.query.strip() or f'{row.artist} {row.title}'
    cands = search_youtube(query)
    if not cands:
        return None, None
    best = max(cands, key=lambda e: score_candidate(e, ref, row.artist, row.title))
    return best, float(score_candidate(best, ref, row.artist, row.title))

def download_clip(video_id, track_id):
    url = f'https://www.youtube.com/watch?v={video_id}'
    tmpl = str(AUDIO_DIR / f'_full_{track_id}.%(ext)s')
    with yt_dlp.YoutubeDL(ydl_opts(format='bestaudio[ext=m4a]/bestaudio/best', outtmpl=tmpl)) as ydl:
        info = ydl.extract_info(url, download=True)
    full = (info.get('requested_downloads') or [{}])[0].get('filepath')
    if not full or not Path(full).exists():
        full = str(next(AUDIO_DIR.glob(f'_full_{track_id}.*')))
    dur = float(info.get('duration') or 0)
    start = max(0.0, min(dur * CLIP_POS, dur - CLIP_SEC)) if dur > CLIP_SEC else 0.0
    out = AUDIO_DIR / f'{track_id}.wav'
    subprocess.run([FFMPEG, '-y', '-loglevel', 'error', '-ss', f'{start:.2f}', '-t', str(CLIP_SEC),
                    '-i', full, '-ac', '1', '-ar', '48000', str(out)], check=True)
    Path(full).unlink(missing_ok=True)
    return info, start

def youtube_record(row, info, start, score):
    vid = info['id']
    up = info.get('upload_date') or ''
    channel = info.get('channel') or info.get('uploader') or ''
    return {
        'track_id': row.track_id, 'artist': row.artist, 'title': row.title, 'genre': row.genre,
        'year': int(row.year), 'source': 'YouTube',
        'yt_video_id': vid, 'yt_url': f'https://www.youtube.com/watch?v={vid}',
        'clip_url': f'https://www.youtube.com/watch?v={vid}&t={int(start)}s',
        'yt_title': info.get('title'), 'yt_channel': channel,
        'is_topic_channel': channel.lower().endswith(' - topic'),
        'duration_sec': info.get('duration'), 'view_count': info.get('view_count'),
        'like_count': info.get('like_count'),
        'upload_date': f'{up[:4]}-{up[4:6]}-{up[6:]}' if len(up) == 8 else None,
        'yt_categories': '|'.join(info.get('categories') or []),
        'yt_tags': '|'.join((info.get('tags') or [])[:10]),
        'match_score': score, 'clip_start_sec': round(start, 2), 'clip_sec': CLIP_SEC,
        'audio_file': f'audio/{row.track_id}.wav',
    }

In [ ]:
# сюда вписываю id правильного видео, если поиск ошибся: {'rock_01': '...'}
MANUAL_OVERRIDES = {}

CACHE_PATH = DATA_DIR / 'youtube_meta_cache.json'
cache = json.loads(CACHE_PATH.read_text(encoding='utf-8')) if CACHE_PATH.exists() else {}
failed = []

for row in tqdm(list(seed.itertuples(index=False))):
    wav = AUDIO_DIR / f'{row.track_id}.wav'
    cached = cache.get(row.track_id)
    if cached and wav.exists() and (row.track_id not in MANUAL_OVERRIDES
                                    or cached['yt_video_id'] == MANUAL_OVERRIDES[row.track_id]):
        continue
    try:
        cand, score = pick_video(row)
        if cand is None:
            raise RuntimeError('ничего не найдено')
        info, start = download_clip(cand['id'], row.track_id)
        cache[row.track_id] = youtube_record(row, info, start, score)
        CACHE_PATH.write_text(json.dumps(cache, ensure_ascii=False, indent=1), encoding='utf-8')
    except Exception as e:
        failed.append((row.track_id, f'{type(e).__name__}: {str(e)[:200]}'))
    time.sleep(SLEEP_SEC)

tracks = pd.DataFrame([cache[t] for t in seed.track_id if t in cache and (AUDIO_DIR / f'{t}.wav').exists()])
print(f'Загружено {len(tracks)} из {len(seed)}')
for t, err in failed:
    print(t, err)

Видео с самой низкой оценкой совпадения, их проверяю вручную.

In [ ]:
tracks['needs_review'] = tracks['match_score'] < 60
tracks.sort_values('match_score')[['track_id', 'artist', 'title', 'yt_title', 'yt_channel', 'match_score', 'yt_url']].head(15)

### Сверка года с MusicBrainz
Для каждого трека (кроме классики) беру самую раннюю дату релиза. Основным оставляю год из своего списка, а треки с расхождением больше 2 лет смотрю отдельно. У MusicBrainz ограничение в 1 запрос в секунду, поэтому ставлю паузу.

In [ ]:
import requests
ENABLE_MUSICBRAINZ = True
MB_HEADERS = {'User-Agent': 'MusicFeaturesStudy/1.0 (student project)'}

def mb_first_year(artist, title):
    title = re.sub(r'\s*\((feat|ft)\..*?\)', '', title, flags=re.I).replace('"', '')
    artist = artist.replace('"', '')
    q = f'recording:"{title}" AND artist:"{artist}"'
    r = requests.get('https://musicbrainz.org/ws/2/recording', headers=MB_HEADERS, timeout=20,
                     params={'query': q, 'fmt': 'json', 'limit': 25})
    r.raise_for_status()
    years = [int(rec['first-release-date'][:4]) for rec in r.json().get('recordings', [])
             if rec.get('score', 0) >= 90 and str(rec.get('first-release-date', ''))[:4].isdigit()]
    return min(years) if years else None

mb_cache_path = DATA_DIR / 'musicbrainz_cache.json'
mb_cache = json.loads(mb_cache_path.read_text(encoding='utf-8')) if mb_cache_path.exists() else {}
if ENABLE_MUSICBRAINZ:
    for row in tqdm(list(tracks.itertuples(index=False))):
        if row.genre == 'classical' or row.track_id in mb_cache:
            continue
        try:
            mb_cache[row.track_id] = mb_first_year(row.artist, row.title)
        except Exception:
            mb_cache[row.track_id] = None
        time.sleep(1.1)
    mb_cache_path.write_text(json.dumps(mb_cache), encoding='utf-8')

tracks['year_musicbrainz'] = tracks['track_id'].map(mb_cache)
diff = (tracks['year'] - tracks['year_musicbrainz']).abs()
print('Найдено в MusicBrainz:', int(tracks.year_musicbrainz.notna().sum()))
tracks.loc[diff > 2, ['artist', 'title', 'year', 'year_musicbrainz']]

In [ ]:
def era_of(y):
    if y < 1900: return 'до 1900 (классика)'
    if y < 1960: return '1900–1959'
    if y < 1980: return '1960–1979'
    if y < 2000: return '1980–1999'
    if y < 2010: return '2000–2009'
    return '2010+'

tracks['decade'] = (tracks['year'] // 10 * 10).astype(int)
tracks['era'] = tracks['year'].apply(era_of)
tracks['log_views'] = np.log10(tracks['view_count'].fillna(0).astype(float) + 1)

tracks.to_csv(DATA_DIR / 'tracks.csv', index=False)
print(tracks.shape)
tracks.groupby('genre').agg(n=('track_id', 'size'), year_min=('year', 'min'), year_max=('year', 'max'),
                            views_median=('view_count', 'median'), topic_share=('is_topic_channel', 'mean')).rename(index=GENRE_RU)

In [ ]:
tracks[['track_id', 'artist', 'title', 'genre', 'year', 'yt_channel', 'duration_sec', 'view_count', 'clip_url']].head(10)